In [ ]:
import os
import sys
from argparse import ArgumentParser, BooleanOptionalAction
import warnings
import json
import torch
import logging
import random
import numpy as np
import re
import time
import pandas as pd
import datetime as dt
from tqdm import tqdm
import logging.config
from datasets import Dataset
# from transformers.utils import logging
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, EarlyStoppingCallback

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)

cache_path = "/scratch/wadhwa.s/pattern_distillation/"

In [ ]:
m = "/scratch/wadhwa.s/pattern_distillation/pattern_distill_models/trained/cnn/gpt2_mistral7binstruct/checkpoint-1000"
tokenizer = AutoTokenizer.from_pretrained(m)
model = AutoModelForCausalLM.from_pretrained(m, 
                                            cache_dir=cache_path,
                                            # trust_remote_code = remote_code,
                                            local_files_only = True,
                                            device_map="auto")

In [ ]:
df = df = pd.read_csv("/work/frink/shaib.c/pattern_distillation/inference/cnn/gpt2_mistral7b.csv")
df.head()

In [ ]:
df["clm"] = tokenizer.bos_token + df["text"] + " #### [SUMMARY]"

In [ ]:
d_test = Dataset.from_pandas(df.sample(5))

In [ ]:
d_test

In [ ]:
processed = []
gold = []
ids = []
article = []
teacher = []
for ins in tqdm(d_test):
    m_input = ins["clm"]
    inputs = tokenizer(m_input, return_tensors="pt").input_ids.to(device)
    outputs = model.generate(inputs, 
                            max_length=1024, 
                            # do_sample=True, 
                            # top_k=50, 
                            # top_p=0.95, 
                            # temperature=0.7,
                            num_return_sequences=1,
                            use_cache=True,
                            pad_token_id=tokenizer.pad_token_id,
                            # eos_token_id=tokenizer.eos_token_id,
                            # bos_token_id=tokenizer.bos_token_id,
                            # no_repeat_ngram_size=2,
                            # early_stopping=True,
                            # num_beams=5,
                            # length_penalty=1.0,
                            )
    torch.cuda.empty_cache()
    generated_ids = outputs.to('cpu')
    generated_tokens = tokenizer.batch_decode(generated_ids, skip_special_tokens=False)
    processed.append(generated_tokens)
    gold.append(ins["gold_summary"])
    teacher.append(ins["generated_summary"])
    article.append(ins["text"])
    ids.append(ins["id"])
    

In [ ]:
for summ, gold, teacher, article, ids in zip(processed, gold, teacher, article, ids):
    op = summ[0]
    match = re.search(r'\[SUMMARY\]\s*(.*?)\s*\[SUMMARY\]', op.strip())
    if match:
        summary = match.group(1)
    else:
        summary = "No summary found"
    print("Student Summary: ", summary)
    print ("Teacher Summary: ", teacher)
    print("Gold Summary: ", gold)
    print("ID: ", ids)
    print("\n---\n")

In [ ]:
df = pd.read_csv("/work/frink/shaib.c/pattern_distillation/inference/pubmed/gpt2_mixtral.csv")
df.head()

In [ ]:
none_counter = 0
for ix, row in df.iterrows():
    print ("ID: ", row["id"])
    print ("\nStudent Summary: ", row["student"])
    if row["student"] == "None":
        none_counter += 1
    print ("\nTeacher Summary: ", row["teacher_summ"])
    print ("\n-------------------\n")

In [ ]:
x = pd.read_csv('/work/frink/shaib.c/pattern_distillation/original_data/pubmed_summ.csv')

In [ ]:
x.token_length.mean()

In [ ]:
len(x)

In [ ]:
x

In [ ]:
5000 * 634 

In [ ]:
3170000 /1000000

In [ ]:
3.17 * 0.05

In [ ]:
x = pd.read_csv('/work/frink/shaib.c/pattern_distillation/original_data/pubmed_summ.csv')

In [ ]:
x.token_length.mean()

In [ ]:
len(x)

In [ ]:
x

In [ ]:
5000 * 634 

In [ ]:
3170000 /1000000

In [ ]:
3.17 * 0.05

In [ ]:
none_counter

In [ ]:
df.shape